In [0]:
from pyspark.sql.functions import col, count, when, isnan, min, max, avg, percentile_approx, when

In [0]:
#Read file
base_path = "/Volumes/clinical_trials/raw/clinical_trials_raw"

#Load studies
studies = spark.read.option("header", "true").option("sep", "|").csv(f"{base_path}/studies.txt")

#Print schema
print("Studies schema:")
studies.printSchema()

Studies schema:
root
 |-- nct_id: string (nullable = true)
 |-- nlm_download_date_description: string (nullable = true)
 |-- study_first_submitted_date: string (nullable = true)
 |-- results_first_submitted_date: string (nullable = true)
 |-- disposition_first_submitted_date: string (nullable = true)
 |-- last_update_submitted_date: string (nullable = true)
 |-- study_first_submitted_qc_date: string (nullable = true)
 |-- study_first_posted_date: string (nullable = true)
 |-- study_first_posted_date_type: string (nullable = true)
 |-- results_first_submitted_qc_date: string (nullable = true)
 |-- results_first_posted_date: string (nullable = true)
 |-- results_first_posted_date_type: string (nullable = true)
 |-- disposition_first_submitted_qc_date: string (nullable = true)
 |-- disposition_first_posted_date: string (nullable = true)
 |-- disposition_first_posted_date_type: string (nullable = true)
 |-- last_update_submitted_qc_date: string (nullable = true)
 |-- last_update_posted_dat

In [0]:
#Inspect status column
print(f"Total studies: {studies.count():,}")
print(f"\nOverall status distribution:")
studies.groupBy("overall_status").count().orderBy("count", ascending=False).show(20)

Total studies: 587,788

Overall status distribution:
+--------------------+------+
|      overall_status| count|
+--------------------+------+
|           COMPLETED|321187|
|             UNKNOWN| 93743|
|          RECRUITING| 64564|
|          TERMINATED| 33698|
|  NOT_YET_RECRUITING| 27679|
|ACTIVE_NOT_RECRUI...| 21608|
|           WITHDRAWN| 16427|
|ENROLLING_BY_INVI...|  5135|
|           SUSPENDED|  1722|
|            WITHHELD|   975|
| NO_LONGER_AVAILABLE|   527|
|           AVAILABLE|   249|
|APPROVED_FOR_MARK...|   237|
|TEMPORARILY_NOT_A...|    37|
+--------------------+------+



In [0]:
#Inspect phase and study type columns
print("Phase distribution:")
studies.groupBy("phase").count().orderBy("count", ascending=False).show(20)

print("Study type distribution:")
studies.groupBy("study_type").count().orderBy("count", ascending=False).show()

Phase distribution:
+-------------+------+
|        phase| count|
+-------------+------+
|           NA|228612|
|         NULL|139389|
|       PHASE2| 64432|
|       PHASE1| 47857|
|       PHASE3| 41693|
|       PHASE4| 35269|
|PHASE1/PHASE2| 16743|
|PHASE2/PHASE3|  7488|
| EARLY_PHASE1|  6305|
+-------------+------+

Study type distribution:
+---------------+------+
|     study_type| count|
+---------------+------+
| INTERVENTIONAL|448515|
|  OBSERVATIONAL|137248|
|EXPANDED_ACCESS|  1050|
|           NULL|   975|
+---------------+------+



In [0]:
key_cols = ["nct_id", "start_date", "completion_date", "overall_status", 
            "phase", "study_type", "enrollment", "source_class"]

print("Null counts on key columns:")
display(studies.select([
    count(when(col(c).isNull() | (col(c) == ""), c)).alias(c) 
    for c in key_cols
]))

Null counts on key columns:


nct_id,start_date,completion_date,overall_status,phase,study_type,enrollment,source_class
0,5339,16701,0,139389,975,7110,975


In [0]:
#Explore calculated values data
calc = spark.read.option("header", "true").option("sep", "|").csv(f"{base_path}/calculated_values.txt")

print(f"Total rows: {calc.count():,}")
print("\nSchema:")
calc.printSchema()

Total rows: 587,788

Schema:
root
 |-- id: string (nullable = true)
 |-- nct_id: string (nullable = true)
 |-- number_of_facilities: string (nullable = true)
 |-- number_of_nsae_subjects: string (nullable = true)
 |-- number_of_sae_subjects: string (nullable = true)
 |-- registered_in_calendar_year: string (nullable = true)
 |-- nlm_download_date: string (nullable = true)
 |-- actual_duration: string (nullable = true)
 |-- were_results_reported: string (nullable = true)
 |-- months_to_report_results: string (nullable = true)
 |-- has_us_facility: string (nullable = true)
 |-- has_single_facility: string (nullable = true)
 |-- minimum_age_num: string (nullable = true)
 |-- maximum_age_num: string (nullable = true)
 |-- minimum_age_unit: string (nullable = true)
 |-- maximum_age_unit: string (nullable = true)
 |-- number_of_primary_outcomes_to_measure: string (nullable = true)
 |-- number_of_secondary_outcomes_to_measure: string (nullable = true)
 |-- number_of_other_outcomes_to_measure:

In [0]:
#Null check and distribution of key ML columns

key_cols = ["nct_id", "actual_duration", "number_of_facilities", 
            "were_results_reported", "months_to_report_results",
            "minimum_age_num", "maximum_age_num"]

print("Null counts:")
calc.select([
    count(when(col(c).isNull() | (col(c) == ""), c)).alias(c) 
    for c in key_cols
]).show()

print("\nDuration stats:")
calc.select(
    min("actual_duration").alias("min"),
    max("actual_duration").alias("max"),
    avg("actual_duration").alias("avg")
).show()

Null counts:
+------+---------------+--------------------+---------------------+------------------------+---------------+---------------+
|nct_id|actual_duration|number_of_facilities|were_results_reported|months_to_report_results|minimum_age_num|maximum_age_num|
+------+---------------+--------------------+---------------------+------------------------+---------------+---------------+
|     0|         227543|                   0|                    0|                  509428|          38451|         275717|
+------+---------------+--------------------+---------------------+------------------------+---------------+---------------+


Duration stats:
+---+---+------------------+
|min|max|               avg|
+---+---+------------------+
|  0| 99|26.942905522630433|
+---+---+------------------+



In [0]:
conditions = spark.read.option("header", "true").option("sep", "|").csv(f"{base_path}/conditions.txt")
sponsors = spark.read.option("header", "true").option("sep", "|").csv(f"{base_path}/sponsors.txt")

print("Conditions schema:")
conditions.printSchema()

print("Sponsors schema:")
sponsors.printSchema()

print("Sponsor class distribution:")
display(sponsors.groupBy("lead_or_collaborator", "agency_class").count().orderBy("count", ascending=False))

Conditions schema:
root
 |-- id: string (nullable = true)
 |-- nct_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- downcase_name: string (nullable = true)

Sponsors schema:
root
 |-- id: string (nullable = true)
 |-- nct_id: string (nullable = true)
 |-- agency_class: string (nullable = true)
 |-- lead_or_collaborator: string (nullable = true)
 |-- name: string (nullable = true)

Sponsor class distribution:


lead_or_collaborator,agency_class,count
lead,OTHER,419303
collaborator,OTHER,186985
lead,INDUSTRY,129964
collaborator,INDUSTRY,53850
collaborator,UNKNOWN,47809
collaborator,NIH,35716
collaborator,OTHER_GOV,17374
lead,OTHER_GOV,15508
lead,NIH,11532
collaborator,FED,5995


Exploring key predictive columns and how trial duration varies across their different values.

In [0]:
# Join the two core tables
joined = studies.join(calc, on="nct_id", how="inner")

# Cast actual_duration to integer first
joined = joined.withColumn("actual_duration", col("actual_duration").cast("int"))

print("Average and median duration by study phase:")
display(
    joined.groupBy("phase")
    .agg(
        count("actual_duration").alias("count"),
        avg("actual_duration").alias("avg_duration_months"),
        percentile_approx("actual_duration", 0.5).alias("median_duration_months")
    )
    .orderBy("median_duration_months", ascending=False)
)

Average and median duration by study phase:


phase,count,avg_duration_months,median_duration_months
PHASE1/PHASE2,9533,36.941151788524074,30
PHASE2,40791,33.47731117158197,26
PHASE3,28076,31.735681721042884,24
PHASE2/PHASE3,4423,31.04024417815962,24
PHASE4,22963,27.055306362409095,21
null,75723,30.577301480395654,19
EARLY_PHASE1,3277,26.062252059810803,19
NA,140032,22.915512168647165,16
PHASE1,35427,20.578231292517007,11


In [0]:
# Join with lead sponsors
lead_sponsors = sponsors.filter(col("lead_or_collaborator") == "lead")
joined2 = joined.join(lead_sponsors, on="nct_id", how="left")

print("Average and median duration by sponsor class:")
display(
    joined2.groupBy("agency_class")
    .agg(
        count("actual_duration").alias("count"),
        avg("actual_duration").alias("avg_duration_months"),
        percentile_approx("actual_duration", 0.5).alias("median_duration_months")
    )
    .orderBy("median_duration_months", ascending=False)
)

print("Average and median duration by study type:")
display(
    joined2.groupBy("study_type")
    .agg(
        count("actual_duration").alias("count"),
        avg("actual_duration").alias("avg_duration_months"),
        percentile_approx("actual_duration", 0.5).alias("median_duration_months")
    )
    .orderBy("median_duration_months", ascending=False)
)

Average and median duration by sponsor class:


agency_class,count,avg_duration_months,median_duration_months
NIH,5758,55.71118443904133,45
NETWORK,3462,47.18630849220104,37
FED,3370,35.88724035608308,33
AMBIG,3,36.333333333333336,27
INDIV,348,31.632183908045977,24
OTHER,243877,28.40020994189694,20
OTHER_GOV,7732,24.11627004655975,16
INDUSTRY,95648,20.66641226162596,15
UNKNOWN,47,11.063829787234043,5
null,0,null,null


Average and median duration by study type:


study_type,count,avg_duration_months,median_duration_months
EXPANDED_ACCESS,30,45.733333333333334,38
INTERVENTIONAL,284637,25.973682268995248,19
OBSERVATIONAL,75578,30.58567307946757,19
null,0,null,null


In [0]:
joined3 = joined.withColumn("facilities_bucket", 
    when(col("number_of_facilities").cast("int") == 1, "1 facility")
    .when(col("number_of_facilities").cast("int") <= 5, "2-5 facilities")
    .when(col("number_of_facilities").cast("int") <= 20, "6-20 facilities")
    .when(col("number_of_facilities").cast("int") <= 100, "21-100 facilities")
    .otherwise("100+ facilities")
)

print("Average and median duration by number of facilities:")
display(
    joined3.groupBy("facilities_bucket")
    .agg(
        count("actual_duration").alias("count"),
        avg("actual_duration").alias("avg_duration_months"),
        percentile_approx("actual_duration", 0.5).alias("median_duration_months")
    )
    .orderBy("median_duration_months", ascending=False)
)

Average and median duration by number of facilities:


facilities_bucket,count,avg_duration_months,median_duration_months
100+ facilities,4594,44.71441010013061,36
6-20 facilities,27819,33.435242100722526,27
21-100 facilities,18472,34.32665656128194,26
2-5 facilities,69003,29.936973754764285,21
1 facility,240357,24.42479728071161,16


In [0]:
print("Were results reported distribution:")
display(
    joined.groupBy("were_results_reported")
    .agg(
        count("actual_duration").alias("count"),
        avg("actual_duration").alias("avg_duration_months"),
        percentile_approx("actual_duration", 0.5).alias("median_duration_months")
    )
)

print("\nEnrollment type distribution (ACTUAL vs ESTIMATED):")
display(
    joined.groupBy("enrollment_type")
    .agg(
        count("actual_duration").alias("count"),
        avg("actual_duration").alias("avg_duration_months"),
        percentile_approx("actual_duration", 0.5).alias("median_duration_months")
    )
)

Were results reported distribution:


were_results_reported,count,avg_duration_months,median_duration_months
t,78312,31.302763305751352,24
f,281933,25.73187601309531,17



Enrollment type distribution (ACTUAL vs ESTIMATED):


enrollment_type,count,avg_duration_months,median_duration_months
null,3278,36.79469188529591,27
ESTIMATED,16339,30.944427443539997,22
ACTUAL,340628,26.656155688904025,18
